# Exercise: How many nodes?

Can a neural net tell **EDM** from **rap**? Do more nodes help?

Let's revisit the Spotify data from the lecture 13 exercise and compare logistic regression, a random forest, and neural nets with 2, 10, and 50 hidden nodes.

## Setup

In [ ]:
library('tidymodels')

options(repr.plot.res = 250, repr.plot.height = 3, repr.plot.width = 5)

## The data

Each row is one song. `genre` is `rap` or `edm`. The predictors are `danceability` and `tempo`.

In [ ]:
songs <- read.csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-01-21/spotify_songs.csv')

songs <- songs |>
    filter(playlist_genre %in% c('edm', 'rap')) |>
    distinct(track_id, .keep_all = TRUE) |>
    mutate(genre = factor(playlist_genre, levels = c('rap', 'edm'))) |>
    select(track_name, track_artist, genre, danceability, tempo)

songs |> count(genre)

## Setup for the models

Run these cells. Do not change them.

- Split the songs in half: `train` and `test`.
- `train_small` has only 100 songs from `train`.
- `fit_nn()` fits a neural net with `h` hidden nodes. It normalizes the predictors first.
- `accuracy_of()` finds the share of songs a model gets right.

In [ ]:
set.seed(512)
split <- initial_split(songs, prop = 0.5, strata = genre)
train <- training(split)
test  <- testing(split)

set.seed(3)
train_small <- train |> slice_sample(n = 100)

In [ ]:
rec <- recipe(genre ~ danceability + tempo, data = songs) |>
    step_normalize(all_predictors())

fit_nn <- function(h, data) {
    set.seed(1)
    workflow() |>
        add_recipe(rec) |>
        add_model(
            mlp(hidden_units = h, epochs = 1000, penalty = 0) |>
                set_mode('classification') |>
                set_engine('nnet', MaxNWts = 5000)
        ) |>
        fit(data = data)
}

accuracy_of <- function(fit, data) {
    pred <- predict(fit, new_data = data, type = 'class')$.pred_class
    mean(pred == data$genre)
}

## 1. Small data: 100 songs

`compare_models()` fits five models: logistic regression, a random forest, and neural nets with 2, 10, and 50 hidden nodes. It returns two accuracies for each model:

- `train_accuracy`: on the songs used to fit the model.
- `test_accuracy`: on the `test` songs, which the model has not seen.

Run the cell. It uses only `train_small`, which has 100 songs.

In [ ]:
compare_models <- function(train_data) {
    set.seed(512)
    fits <- list(
        logistic = logistic_reg() |> fit(genre ~ danceability + tempo, data = train_data),
        forest   = rand_forest(trees = 200) |> set_mode('classification') |> set_engine('ranger') |>
                       fit(genre ~ danceability + tempo, data = train_data),
        `nn 2`   = fit_nn(2, train_data),
        `nn 10`  = fit_nn(10, train_data),
        `nn 50`  = fit_nn(50, train_data)
    )

    tibble(
        model = names(fits),
        train_accuracy = map_dbl(fits, accuracy_of, data = train_data),
        test_accuracy  = map_dbl(fits, accuracy_of, data = test)
    )
}

results_small = compare_models(train_small)

results_small

**Check:** `nn 50` should be right for about 0.99 of the training songs.

- Which model is best on the **training** songs? Which is best on the **test** songs?
- What happens to the test accuracy of the nets as the number of nodes goes up?
- A net with 50 hidden nodes learns 201 numbers. How many songs did we give it?

## 2. Chart the output

Pick a visualization for the output and make the chart below. Write the code by hand or prompt it with `OpenCode`.

In [ ]:
# Chart code goes here



## 3. More data

Run `compare_models()` again, with all of `train` (about 5,500 songs).

**Check:** the nets should now be close to the forest, at about 0.82 on the test songs.

- Is `nn 50` still much worse than `nn 2`? Do more nodes help now?
- Logistic regression is stuck at about 0.60. Why? (Think about the stripe in the lecture 13 exercise.)

## 4. Plot the results when using all the data

In [ ]:
# Chart code goes here



## 5. Which model?

You must pick one model for a new playlist with **100 labeled songs**. Which do you pick?

*Your answer:*